In [ ]:
import pandas as pd
import os
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings

CHROMA_BASE_PATH = "./2k/chrome_db"
DATA_BASE_PATH = "./2k"
ANIME_MsETADATA_PATH = "./raw/anime_filtered.csv" 

TARGET_USER = 'abystoma2'
LIKED_ANIME = 'One Piece'

In [2]:
users_df = pd.read_parquet(os.path.join(DATA_BASE_PATH, "users_sample.parquet"))
animelist_df = pd.read_parquet(os.path.join(DATA_BASE_PATH, "animelist_sample.parquet"))
anime_df = pd.read_csv(ANIME_METADATA_PATH)

users_df['username_lower'] = users_df['username'].str.lower()
anime_df['title_lower'] = anime_df['title'].str.lower()
animelist_df['username_lower'] = animelist_df['username'].str.lower()

In [3]:
embeddings = OllamaEmbeddings(model="nomic-embed-text")

retriever_users = Chroma(
    collection_name="user_profiles",
    persist_directory=os.path.join(CHROMA_BASE_PATH, "users"),
    embedding_function=embeddings
).as_retriever(search_kwargs={"k": 20})

retriever_anime = Chroma(
    collection_name="anime_profiles",
    persist_directory=os.path.join(CHROMA_BASE_PATH, "animes"),
    embedding_function=embeddings
).as_retriever(search_kwargs={"k": 10})

retriever_interactions = Chroma(
    collection_name="anime_interactions",
    persist_directory=os.path.join(CHROMA_BASE_PATH, "anime_user"),
    embedding_function=embeddings
).as_retriever(search_kwargs={"k": 30})

In [5]:
anime_df.head()

,anime_id,title,title_english,title_japanese,title_synonyms,image_url,type,source,episodes,status,...,premiered,broadcast,related,producer,licensor,studio,genre,opening_theme,ending_theme,title_lower
0,11013,Inu x Boku SS,Inu X Boku Secret Service,妖狐×僕SS,Youko x Boku SS,https://myanimelist.cdn-dena.com/images/anime/...,TV,Manga,12,Finished Airing,...,Winter 2012,Fridays at Unknown,"{'Adaptation': [{'mal_id': 17207, 'type': 'man...","Aniplex, Square Enix, Mainichi Broadcasting Sy...",Sentai Filmworks,David Production,"Comedy, Supernatural, Romance, Shounen","['""Nirvana"" by MUCC']","['#1: ""Nirvana"" by MUCC (eps 1, 11-12)', '#2: ...",inu x boku ss
1,2104,Seto no Hanayome,My Bride is a Mermaid,瀬戸の花嫁,The Inland Sea Bride,https://myanimelist.cdn-dena.com/images/anime/...,TV,Manga,26,Finished Airing,...,Spring 2007,Unknown,"{'Adaptation': [{'mal_id': 759, 'type': 'manga...","TV Tokyo, AIC, Square Enix, Sotsu",Funimation,Gonzo,"Comedy, Parody, Romance, School, Shounen","['""Romantic summer"" by SUN&LUNAR']","['#1: ""Ashita e no Hikari (明日への光)"" by Asuka Hi...",seto no hanayome
2,5262,Shugo Chara!! Doki,Shugo Chara!! Doki,しゅごキャラ！！どきっ,"Shugo Chara Ninenme, Shugo Chara! Second Year",https://myanimelist.cdn-dena.com/images/anime/...,TV,Manga,51,Finished Airing,...,Fall 2008,Unknown,"{'Adaptation': [{'mal_id': 101, 'type': 'manga...","TV Tokyo, Sotsu",NaN,Satelight,"Comedy, Magic, School, Shoujo","['#1: ""Minna no Tamago (みんなのたまご)"" by Shugo Cha...","['#1: ""Rottara Rottara (ロッタラ ロッタラ)"" by Buono! ...",shugo chara!! doki
3,721,Princess Tutu,Princess Tutu,プリンセスチュチュ,NaN,https://myanimelist.cdn-dena.com/images/anime/...,TV,Original,38,Finished Airing,...,Summer 2002,Fridays at Unknown,"{'Adaptation': [{'mal_id': 1581, 'type': 'mang...","Memory-Tech, GANSIS, Marvelous AQL",ADV Films,Hal Film Maker,"Comedy, Drama, Magic, Romance, Fantasy","['""Morning Grace"" by Ritsuko Okazaki']","['""Watashi No Ai Wa Chiisaikeredo"" by Ritsuko ...",princess tutu
4,12365,Bakuman. 3rd Season,Bakuman.,バクマン。,Bakuman Season 3,https://myanimelist.cdn-dena.com/images/anime/...,TV,Manga,25,Finished Airing,...,Fall 2012,Unknown,"{'Adaptation': [{'mal_id': 9711, 'type': 'mang...","NHK, Shueisha",NaN,J.C.Staff,"Comedy, Drama, Romance, Shounen","['#1: ""Moshimo no Hanashi (もしもの話)"" by nano.RIP...","['#1: ""Pride on Everyday"" by Sphere (eps 1-13)...",bakuman. 3rd season


In [ ]:
# Esta é a função corrigida. Note a nova linha pd.merge no início.
def get_recommendations_from_users(usernames: list, exclude_username: str, top_n: int = 30) -> list:
    """
    Suggests animes that a group of users liked, excluding animes already seen by a target user.
    """
    # ETAPA DE CORREÇÃO: Juntamos o animelist_df com o anime_df para ter acesso aos títulos.
    # Usamos apenas as colunas necessárias para a operação ser mais rápida.
    merged_df = pd.merge(animelist_df, anime_df[['anime_id', 'title']], on='anime_id', how='left')

    # 1. Encontra todos os animes que o usuário de referência já assistiu
    target_user_watched = set(
        merged_df[merged_df['username_lower'] == exclude_username.lower()]['title']
    )

    # 2. Pega todas as avaliações altas (>= 8) do grupo de usuários similares
    recs_df = merged_df[
        (merged_df['username'].isin(usernames)) &
        (merged_df['my_score'] >= 5)
    ]
    
    # 3. Remove os animes que o usuário de referência já viu
    #recs_df = recs_df[~recs_df['title'].isin(target_user_watched)]
    if recs_df.empty:
        return []
    
    # 4. Conta quais animes aparecem mais vezes, ordena e retorna os N melhores
    top_recs = recs_df.groupby('title').size().sort_values(ascending=False).head(top_n)
    
    return top_recs.index.tolist()

def find_recommendations_from_opinion_peers(username: str, anime_title: str, k: int = 10) -> list:
    """
    Finds recommended animes from a group of users who share a similar opinion
    about a specific anime.

    Args:
        username: The reference user.
        anime_title: The title of the anime to base the opinion on.
        k: The number of final anime recommendations to return.

    Returns:
        A list of recommended anime titles.
    """
    try:
        # --- Etapa 1: Encontrar o anime de referência ---
        anime_row = anime_df[anime_df['title'].str.lower() == anime_title.lower()].iloc[0]
        anime_id = anime_row['anime_id']
        
        # --- Etapa 2: Construir a query baseada na opinião do usuário de referência ---
        interaction_row = animelist_df[
            (animelist_df['username_lower'] == username.lower()) & 
            (animelist_df['anime_id'] == anime_id)
        ].iloc[0]

        score = interaction_row.get('my_score', 0)
        status_code = interaction_row.get('my_status', 0)
        tags = interaction_row.get('my_tags', 'no tags')
        status_map = {1: "Watching", 2: "Completed", 3: "On-Hold", 4: "Dropped", 6: "Plan to Watch"}
        status_text = status_map.get(status_code, "Unknown Status")
        score_text = f"Rated this anime {score} out of 10." if score > 0 else "This anime is unrated."
        query_text = f"{score_text} Status is {status_text}. User tags: {tags}"
        
        # --- Etapa 3: Encontrar a "tribo" de usuários com opinião similar ---
        similar_docs = retriever_interactions.invoke(query_text)
        
        similar_usernames = list(set([
            doc.metadata['username'] for doc in similar_docs 
            if doc.metadata['username'].lower() != username.lower()
        ]))
        
        if not similar_usernames:
            return [{"error": "No users with a similar opinion were found."}]

        # --- ETAPA NOVA E FINAL: Buscar recomendações a partir dessa "tribo" ---
        # Usamos a ferramenta que já tínhamos para pegar as recomendações desse grupo.

        recommendations = get_recommendations_from_users(
            usernames=similar_usernames, 
            exclude_username=username, 
            top_n=k
        )
        
        return recommendations

    except IndexError:
        return [{"error": f"Could not find an interaction for user '{username}' with anime '{anime_title}'."}]
    except Exception as e:
        return [{"error": f"An error occurred: {e}"}]

# --- Exemplo de Uso ---
print("--- Buscando recomendações de animes a partir de usuários com opinião similar sobre 'One Piece' ---")

# A função agora retorna uma lista de TÍTULOS DE ANIMES
recommended_animes = find_recommendations_from_opinion_peers('abystoma2', 'One Piece', k=10)

print("\nAnimes recomendados:")
print(recommended_animes)

--- Buscando recomendações de animes a partir de usuários com opinião similar sobre 'One Piece' ---

Animes recomendados:
['Sword Art Online', 'Angel Beats!', 'Shingeki no Kyojin', 'Mirai Nikki (TV)', 'Code Geass: Hangyaku no Lelouch', 'Elfen Lied', 'Another', 'Guilty Crown', 'Steins;Gate', 'Clannad']


In [5]:
# Assume que as seguintes variáveis já estão carregadas:
# - DataFrames: users_df, animelist_df, anime_df
# - Retrievers: retriever_users
# - Funções de ajuda: get_recommendations_from_users

def find_recs_from_profile_peers_who_watched_anime(
    target_username: str, 
    watched_anime_title: str, 
    k_similar_users: int = 50, 
    k_final_recs: int = 10
) -> list:
    """
    Finds anime recommendations using a three-step filtering process:
    1. Finds users with profiles similar to the target user.
    2. Filters that list to keep only those who also watched a specific anime.
    3. Gets recommendations from that final, highly-relevant group.

    Args:
        target_username: The reference user.
        watched_anime_title: The anime that the peer group must have watched.
        k_similar_users: The initial number of similar profiles to retrieve. A larger number increases the chance of finding matches.
        k_final_recs: The number of final anime recommendations to return.

    Returns:
        A list of recommended anime titles.
    """
    try:
        # --- ETAPA 1: Encontrar usuários com perfis similares ---
        user_row = users_df[users_df['username_lower'] == target_username.lower()].iloc[0]
        user_query_text = f"User from {user_row.get('location', 'Unknown')}. Gender {user_row.get('gender', 'Unknown')}. Born in {pd.to_datetime(user_row['birth_date']).year}."
        
        retriever_users.search_kwargs['k'] = k_similar_users
        similar_profile_docs = retriever_users.invoke(user_query_text)
        similar_profile_users = set(doc.metadata['username'] for doc in similar_profile_docs)

        if not similar_profile_users:
            return [{"error": "No users with a similar profile were found."}]

        # --- ETAPA 2: Desses usuários, encontrar quem assistiu 'One Piece' ---
        anime_row = anime_df[anime_df['title_lower'] == watched_anime_title.lower()].iloc[0]
        anime_id = anime_row['anime_id']
        
        # Filtra o animelist para o anime específico
        watchers_df = animelist_df[animelist_df['anime_id'] == anime_id]
        
        # Pega a lista de todos que viram One Piece
        all_watchers = set(watchers_df['username'])
        
        # A interseção nos dá nosso grupo de elite
        final_peer_group = list(similar_profile_users.intersection(all_watchers))

        if not final_peer_group:
            return [{"error": f"No users with a similar profile to '{target_username}' also watched '{watched_anime_title}'."}]

        # --- ETAPA 3: Obter recomendações desse grupo final ---
        recommendations = get_recommendations_from_users(
            usernames=final_peer_group,
            exclude_username=target_username,
            top_n=k_final_recs
        )

        return recommendations

    except IndexError:
        return [{"error": f"Could not find data for user '{target_username}' or anime '{watched_anime_title}'."}]
    except Exception as e:
        return [{"error": f"An error occurred: {e}"}]

# --- Exemplo de Uso ---
print(f"--- Buscando recomendações para 'abystoma2' de usuários com perfil similar que também viram 'One Piece' ---")

# Usamos um k_similar_users maior (ex: 50) para aumentar a chance de encontrar sobreposição
recommendations = find_recs_from_profile_peers_who_watched_anime(
    target_username='abystoma2', 
    watched_anime_title='One Piece',
    k_similar_users=50, # Pega um grupo maior de perfis similares
    k_final_recs=50     # Pede as 10 melhores recomendações no final
)

print("\nAnimes recomendados por este grupo:")
print(recommendations)

--- Buscando recomendações para 'abystoma2' de usuários com perfil similar que também viram 'One Piece' ---

Animes recomendados por este grupo:
[{'error': 'No users with a similar profile were found.'}]


In [6]:
import pandas as pd

def get_user_profile(username: str) -> dict:
    """
    Busca e retorna o perfil completo de um usuário.
    
    Args:
        username: O nome do usuário a ser buscado.
        
    Returns:
        Um dicionário com os dados do usuário ou uma mensagem de erro.
    """
    try:
        # Usa a coluna em minúsculas para uma busca insensível ao caso
        user_data = users_df[users_df['username_lower'] == username.lower()].iloc[0]
        # Converte a linha do DataFrame em um dicionário para fácil manipulação
        return user_data.to_dict()
    except IndexError:
        # Retorna um erro se o usuário não for encontrado no DataFrame
        return {"error": f"Usuário '{username}' não foi encontrado."}

# --- Exemplo de Uso ---
user_profile = get_user_profile('abystoma2')

print("--- Perfil do Usuário 'abystoma2' ---")
print(user_profile)

--- Perfil do Usuário 'abystoma2' ---
{'username': 'abystoma2', 'user_id': 3336425, 'user_watching': 180, 'user_completed': 5121, 'user_onhold': 296, 'user_dropped': 0, 'user_plantowatch': 969, 'user_days_spent_watching': 629.24, 'gender': 'Male', 'location': 'Czech Republic', 'birth_date': '1997-06-02', 'access_rank': nan, 'join_date': '2013-11-14', 'last_online': '2018-05-18 01:22:00', 'stats_mean_score': 4.74, 'stats_rewatched': 147.0, 'stats_episodes': 44151.0, 'username_lower': 'abystoma2'}


In [7]:
import pandas as pd

def get_anime_details(anime_title: str) -> dict:
    """
    Busca e retorna os detalhes completos de um anime.
    
    Args:
        anime_title: O título do anime a ser buscado.
        
    Returns:
        Um dicionário com os dados do anime ou uma mensagem de erro.
    """
    try:
        # Usa a coluna em minúsculas para uma busca insensível ao caso
        anime_data = anime_df[anime_df['title_lower'] == anime_title.lower()].iloc[0]
        # Converte a linha do DataFrame em um dicionário
        return anime_data.to_dict()
    except IndexError:
        # Retorna um erro se o anime não for encontrado no DataFrame
        return {"error": f"Anime '{anime_title}' não foi encontrado."}

# --- Exemplo de Uso ---
anime_details = get_anime_details('One Piece')

print("\n--- Detalhes do Anime 'One Piece' ---")
print(anime_details)


--- Detalhes do Anime 'One Piece' ---
{'anime_id': 21, 'title': 'One Piece', 'title_english': 'One Piece', 'title_japanese': 'ONE PIECE', 'title_synonyms': 'OP', 'image_url': 'https://myanimelist.cdn-dena.com/images/anime/6/73245.jpg', 'type': 'TV', 'source': 'Manga', 'episodes': 0, 'status': 'Currently Airing', 'airing': True, 'aired_string': 'Oct 20, 1999 to ?', 'aired': "{'from': '1999-10-20', 'to': None}", 'duration': '24 min.', 'rating': 'PG-13 - Teens 13 or older', 'score': 8.54, 'scored_by': 423868, 'rank': 91.0, 'popularity': 35, 'members': 720133, 'favorites': 69760, 'background': 'Several anime-original arcs have been adapted into light novels, and the series has inspired 40 video games as of 2016.', 'premiered': 'Fall 1999', 'broadcast': 'Sundays at 09:30 (JST)', 'related': "{'Adaptation': [{'mal_id': 13, 'type': 'manga', 'url': 'https://myanimelist.net/manga/13/One_Piece', 'title': 'One Piece'}, {'mal_id': 94534, 'type': 'manga', 'url': 'https://myanimelist.net/manga/94534

In [8]:
import pandas as pd

def find_similar_animes(anime_title: str, k: int = 5) -> list:
    """
    Finds animes with similar content using vector similarity search.

    Args:
        anime_title: The title of the anime to find similar items for.
        k: The number of similar animes to return.

    Returns:
        A list of dictionaries, where each dictionary contains the details
        of a recommended anime, or a list with an error message.
    """
    try:
        # --- 1. Encontrar os dados do anime de referência ---
        anime_row = anime_df[anime_df['title_lower'] == anime_title.lower()].iloc[0]

        # --- 2. Construir o texto da query (deve ser idêntico ao formato do embedding) ---
        title = anime_row.get('title_english') or anime_row.get('title', '')
        genres = anime_row.get('genre', '')
        studio = anime_row.get('studio', '')
        rating_classification = anime_row.get('rating', '')
        background = anime_row.get('background', '')

        query_text = f"Title: {title}. Genres: {genres}. Studio: {studio}. Rating: {rating_classification}. Background: {background}"

        # --- 3. Executar a busca por similaridade ---
        # Buscamos k+1 porque o resultado mais similar é quase sempre o próprio anime.
        retriever_anime.search_kwargs['k'] = k + 1
        similar_docs = retriever_anime.invoke(query_text)

        # --- 4. Processar e formatar os resultados ---
        recommendations = []
        for doc in similar_docs:
            # Garante que não estamos recomendando o próprio anime de entrada
            if doc.metadata.get('title', '').lower() != anime_title.lower():
                recommendations.append({
                    "title": doc.metadata.get('title'),
                    "score": doc.metadata.get('score'),
                    "genres": doc.metadata.get('genre')
                })
        
        # Retorna a quantidade exata de 'k' recomendações
        return recommendations[:k]

    except IndexError:
        return [{"error": f"Anime '{anime_title}' não foi encontrado no dataset."}]
    except Exception as e:
        return [{"error": f"Ocorreu um erro durante a busca: {e}"}]

# --- Exemplo de Uso ---
# Vamos encontrar os 5 animes mais similares a "Fullmetal Alchemist: Brotherhood"
similar_items = find_similar_animes("Fullmetal Alchemist: Brotherhood", k=50)

print("--- Animes Similares a 'Fullmetal Alchemist: Brotherhood' ---")

# Imprime os resultados de forma legível
for item in similar_items:
    if "error" in item:
        print(item["error"])
    else:
        print(f"\n- Título: {item['title']}")
        print(f"  Nota: {item['score']}")
        print(f"  Gêneros: {item['genres']}")

--- Animes Similares a 'Fullmetal Alchemist: Brotherhood' ---


In [9]:
similar_items = find_similar_animes("Great Teacher Onizuka", k=50)
print("similar_items", similar_items)

similar_items []


In [10]:
import pandas as pd

def find_similar_user_profiles(username: str, k: int = 5) -> list:
    """
    Finds users with similar demographic and behavioral profiles using vector search.

    Args:
        username: The username to find similar profiles for.
        k: The number of similar users to return.

    Returns:
        A list of dictionaries, where each dictionary contains the details
        of a similar user, or a list with an error message.
    """
    try:
        # --- 1. Encontrar os dados do usuário de referência ---
        user_row = users_df[users_df['username_lower'] == username.lower()].iloc[0]

        # --- 2. Construir o texto da query (deve ser idêntico ao formato do embedding) ---
        location = user_row.get('location', 'Location Unknown')
        gender = user_row.get('gender', 'Gender Unknown')
        
        try:
            birth_year = pd.to_datetime(user_row['birth_date']).year
            birth_info = f"Born in {birth_year}."
        except (ValueError, TypeError):
            birth_info = "Birth date not specified."
            
        query_text = f"User from {location}. Gender {gender}. {birth_info}"

        # --- 3. Executar a busca por similaridade ---
        # Buscamos k+1 porque o resultado mais similar é sempre o próprio usuário.
        retriever_users.search_kwargs['k'] = k + 1
        similar_docs = retriever_users.invoke(query_text)

        # --- 4. Processar e formatar os resultados ---
        similar_profiles = []
        for doc in similar_docs:
            # Garante que não estamos incluindo o próprio usuário nos resultados
            if doc.metadata.get('username', '').lower() != username.lower():
                similar_profiles.append({
                    "username": doc.metadata.get('username'),
                    "location": doc.metadata.get('location', 'N/A'),
                    "gender": doc.metadata.get('gender', 'N/A'),
                    "mean_score": doc.metadata.get('stats_mean_score')
                })
        
        # Retorna a quantidade exata de 'k' perfis
        return similar_profiles[:k]

    except IndexError:
        return [{"error": f"Usuário '{username}' não foi encontrado no dataset."}]
    except Exception as e:
        return [{"error": f"Ocorreu um erro durante a busca: {e}"}]

# --- Exemplo de Uso ---
# Vamos encontrar os 5 usuários mais similares a "abystoma2"
similar_users = find_similar_user_profiles("abystoma2", k=5)

print("--- Usuários com Perfis Similares a 'abystoma2' ---")

# Imprime os resultados de forma legível
for profile in similar_users:
    if "error" in profile:
        print(profile["error"])
    else:
        print(f"\n- Username: {profile['username']}")
        print(f"  Localização: {profile['location']}")
        print(f"  Gênero: {profile['gender']}")
        print(f"  Nota Média: {profile['mean_score']}")

--- Usuários com Perfis Similares a 'abystoma2' ---
